# Download and test local Planet basemap tiles

Download the nine requested quads for January–September 2025 from the temporary
`planet-data-axis3` bucket, then test parcel statistics without HUB registration.
The delivery layout and credentials follow the downloader used by
`esa_govermental_hub/src/notebooks/upload/upload_batch_process_planet.ipynb`.
This notebook does not execute, edit, or interrupt that ingestion notebook.

The default reuses the completed local download. Set `DOWNLOAD_FILES = True`
to download or resume it. Only S3 listing and object downloads are performed;
there are no upload, registration, or source-deletion operations.

In [1]:
from pathlib import Path

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
import contextlib
import json
import sys
import time

# Find this repository whether launched from its root or this notebook folder.

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from satellites.data_preparation.sources.planet.delivery import (
    create_planet_delivery_client, discover_planet_delivery_tiles,
    download_planet_delivery_tiles, build_local_planet_manifest,
)
from satellites.data_preparation.parcel_stats.planet import PlanetBasemapZonalStats

In [2]:
QUAD_IDS = [
    "1154-1279", "1155-1279", "1154-1278", "1155-1278", "1156-1278",
    "1154-1277", "1155-1277", "1156-1277", "1157-1277",
]
START_DATE = "2023-12-01"
END_DATE = "2024-09-30"
BUCKET = "planet-data-axis3"
DOWNLOAD_DIR = Path("C:/work_dir/planet_local_2025_01_09")
DOWNLOAD_FILES = True  # True: download/resume; False: use existing local imagery only.
INCLUDE_AUXILIARY = True  # Keep UDM2 and provenance alongside the TIFF and metadata.
CREDENTIALS_ENV_FILE = REPO.parent / "esa_govermental_hub/.env"  # Read-only; never copied into outputs.

PARCELS_PATH = Path("C:/work_dir/volvi-2023-2024/parcels_volvi.gpkg")
TILES_PATH = Path("C:/work_dir/planet_basemap_quads.gpkg")
OUTPUT_DIR = Path("C:/work_dir/planet_local_2025_01_09_validation")
WORKING_EPSG = 2100
PARCEL_ID_FIELD = "parcel_code"
SAMPLE_PER_QUAD = 3
SEAM_SAMPLES_PER_GROUP = 2
REFLECTANCE_SCALE = 0.0001  # Verified from the delivered DN values; dtype float32 does not imply reflectance.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEST_ALL_COVERED = True  # True: validate all covered parcels; False: fast representative test.

## Resolve completed deliveries and download

One manifest-backed analysis TIFF and metadata file are required per quad/month.
Each object is checked against the manifest size and SHA256 digest before its
temporary file is renamed. Reruns reuse verified files. Two multipart workers
keep transfer concurrency modest alongside the running ingestion job.

The complete requested set includes 81 TIFFs and 324 metadata/quality/provenance
files (about 28 GB). Download details go to a log rather than filling notebook output.

In [ ]:
import importlib
from satellites.data_preparation.sources.planet import delivery as planet_delivery

# Reload in case delivery.py changed during this session.
planet_delivery = importlib.reload(planet_delivery)
discover_planet_delivery_tiles = planet_delivery.discover_planet_delivery_tiles
download_planet_delivery_tiles = planet_delivery.download_planet_delivery_tiles
create_planet_delivery_client = planet_delivery.create_planet_delivery_client

download_dir_has_rasters = DOWNLOAD_DIR.exists() and any(DOWNLOAD_DIR.rglob("*_quad*.tif"))
allow_partial_download = DOWNLOAD_FILES and not download_dir_has_rasters
used_fallback_all_tiles = False

if DOWNLOAD_FILES:
    client = create_planet_delivery_client(CREDENTIALS_ENV_FILE)
    try:
        with (OUTPUT_DIR / "manual_download.log").open("a", encoding="utf-8") as log, contextlib.redirect_stdout(log):
            try:
                inventory = discover_planet_delivery_tiles(
                    client, BUCKET, QUAD_IDS, START_DATE, END_DATE, require_complete=not allow_partial_download
                )
            except ValueError as error:
                if allow_partial_download and "No completed Planet deliveries found" in str(error):
                    used_fallback_all_tiles = True
                    print(
                        "Requested date range has no completed deliveries in the bucket. "
                        "Falling back to all available months for the requested quads."
                    )
                    inventory = discover_planet_delivery_tiles(
                        client, BUCKET, QUAD_IDS, "2016-01-01", pd.Timestamp.today().strftime("%Y-%m-%d"), require_complete=False
                    )
                else:
                    raise
            manifest = download_planet_delivery_tiles(
                client, BUCKET, inventory, DOWNLOAD_DIR, include_auxiliary=INCLUDE_AUXILIARY
            )
    finally:
        client.close()
else:
    manifest = build_local_planet_manifest(
        DOWNLOAD_DIR, quad_ids=QUAD_IDS, start_date=START_DATE, end_date=END_DATE
    )

# Explicitly verify the requested Cartesian product before processing.
if used_fallback_all_tiles:
    months = sorted(manifest.period_start.unique().tolist())
else:
    months = pd.period_range(START_DATE, END_DATE, freq="M").start_time.strftime("%Y-%m-%d").tolist()
expected = {(month, quad) for month in months for quad in QUAD_IDS}
actual = set(zip(manifest.period_start, manifest.quad_id))
missing = sorted(expected - actual)
extra = sorted(actual - expected)

if allow_partial_download or used_fallback_all_tiles:
    if missing:
        pd.DataFrame(missing, columns=["period_start", "quad_id"]).to_csv(
            OUTPUT_DIR / "missing_requested_tile_months.csv", index=False
        )
        print(
            f"Warning: missing {len(missing)} requested tile/month pairs in bucket for this date range. "
            "Continuing with available deliveries."
        )
    if extra:
        print(f"Warning: found {len(extra)} unexpected tile/month pairs in manifest")
else:
    assert actual == expected and len(manifest) == len(expected), "Missing or duplicated tile/months"

manifest.to_csv(OUTPUT_DIR / "validated_local_manifest.csv", index=False)
print(f"Validated {len(manifest)} available tile/months for processing")
print(manifest.groupby("period_start").size().to_string())

## Check the delivered raster layout

These files have ten bands: eight physical bands, NDVI, and alpha. Their color
interpretation tags alone do not establish spectral order. On a native-resolution
sample, compare supplied band 9 with NDVI from physical B8/B6. Only the first
eight bands feed cleaning; the pipeline recomputes indices and honors alpha.
UDM2 is retained for provenance but is not applied as an additional cloud mask.

In [ ]:
schemas = []
for row in manifest.itertuples():
    with rasterio.open(row.path) as src:
        assert src.count >= 8 and src.crs is not None
        schemas.append({"period_start": row.period_start, "quad_id": row.quad_id,
                        "bands": src.count, "dtype": src.dtypes[0], "crs": str(src.crs),
                        "width": src.width, "height": src.height,
                        "alpha": any(c.name == "alpha" for c in src.colorinterp)})
pd.DataFrame(schemas).to_csv(OUTPUT_DIR / "raster_schema_audit.csv", index=False)
with rasterio.open(manifest.iloc[0].path) as src:
    sample = src.read(window=rasterio.windows.Window(src.width // 2 - 128, src.height // 2 - 128, 256, 256), masked=True)
    expected_ndvi = (sample[7] - sample[5]) / (sample[7] + sample[5])
    ndvi_error = float(np.ma.mean(np.ma.abs(sample[8] - expected_ndvi))) if src.count >= 9 else None
    print("Band count:", src.count, "dtype:", src.dtypes[0], "CRS:", src.crs)
    print("Physical-band median DN:", np.ma.median(sample[:8], axis=(1, 2)).tolist())
    print("Supplied vs B8/B6 NDVI mean absolute difference:", ndvi_error)
    if ndvi_error is not None:
        assert ndvi_error < 1e-5, "Confirm the delivery's spectral band ordering before extraction"

## Select real parcels for the integration test

Use a deterministic sample of three contained parcels per requested quad and up
to two parcels per seam combination. This exercises all nine quads, cross-tile
parcels and all nine months without launching a full 28,699-parcel production run.
Parcels outside the nine supplied quads are counted and excluded explicitly.
Sample members and coverage counts are saved for reproducibility.

Set TEST_ALL_COVERED=True for the complete covered parcel set instead of the sample.

In [ ]:
parcels = gpd.read_file(PARCELS_PATH).to_crs(WORKING_EPSG)
tiles = gpd.read_file(TILES_PATH).to_crs(WORKING_EPSG)
tiles = tiles[tiles.quad_id.astype(str).isin(QUAD_IDS)].sort_values("quad_id").copy()
assert set(tiles.quad_id) == set(QUAD_IDS)
covered = parcels.geometry.covered_by(tiles.geometry.union_all())
eligible = parcels[covered].copy()
parcels.loc[~covered].to_parquet(OUTPUT_DIR / "outside_requested_tiles.geoparquet", index=False)
chosen = set()
contained = set()
coverage_counts = []
for tile in tiles.itertuples():
    candidates = eligible[eligible.geometry.covered_by(tile.geometry)]
    contained.update(candidates.index)
    # Nearby candidates minimize extra spatial batches while representing each quad.
    closest = candidates.geometry.centroid.distance(tile.geometry.centroid).sort_values().head(SAMPLE_PER_QUAD).index
    chosen.update(closest)
    coverage_counts.append({"quad_id": tile.quad_id, "contained_parcels": len(candidates), "sampled": len(closest)})
seam_groups = {}
for index, parcel in eligible.loc[~eligible.index.isin(contained)].iterrows():
    hits = tiles[tiles.geometry.intersects(parcel.geometry)]
    hits = hits[hits.geometry.intersection(parcel.geometry).area > 0]
    group = tuple(sorted(hits.quad_id))
    seam_groups.setdefault(group, []).append(index)
for indices in seam_groups.values():
    chosen.update(indices[:SEAM_SAMPLES_PER_GROUP])
test_parcels = eligible.copy() if TEST_ALL_COVERED else eligible.loc[sorted(chosen)].copy()
test_parcels.to_parquet(OUTPUT_DIR / "test_parcels.geoparquet", index=False)
pd.DataFrame(coverage_counts).to_csv(OUTPUT_DIR / "quad_parcel_coverage.csv", index=False)
print(f"Input: {len(parcels)}; fully covered: {len(eligible)}; outside coverage: {(~covered).sum()}")
print(f"Testing {len(test_parcels)} real parcels, including {len(seam_groups)} seam combinations")
print(pd.DataFrame(coverage_counts).to_string(index=False))

## Run the shared cleaning and statistics pipeline on local files

`run_local` accepts either the directory above or the validated manifest CSV.
It has no catalog or S3 connection. The test enables IQR cleaning and bidirectional
temporal/spatial filling using the existing Planet notebook's settings.
Raw and cleaned test cubes are retained in this separate validation directory;
source downloads are preserved for subsequent experiments.

In [ ]:
extractor = PlanetBasemapZonalStats(
    parcels=test_parcels, start_date=START_DATE, end_date=END_DATE,
    output_dir=OUTPUT_DIR / "parcel_stats", working_epsg=WORKING_EPSG,
    parcel_id_field=PARCEL_ID_FIELD, resolution_metres=5, batch_size_metres=2000,
    buffer_metres=100, max_cube_bytes=512_000_000,
    indices=["EVI", "NDRE", "NDVI", "NDWI", "SAVI"], reflectance_scale=REFLECTANCE_SCALE,
    spatial_statistics=["mean", "median", "sd", "min", "max", "range"],
    remove_outliers=True, iqr_quantiles=(0.1, 0.9), iqr_multiplier=1.5,
    iqr_min_valid_pixels=20, fill_nulls=True, temporal_fill_mode="bidirectional",
    minimum_parcel_pixels=3, minimum_observed_fraction_for_fill=0.20,
    spatial_fill_window_sizes=(3, 5), interpolation_method="nearest",
    interpolation_max_distance_in_meters=500,
)
started = time.monotonic()
result = extractor.run_local(OUTPUT_DIR / "validated_local_manifest.csv")
elapsed_seconds = time.monotonic() - started
print(f"Completed {len(result)} parcels in {elapsed_seconds:.1f} seconds")

## Validate outputs and record the result

Verify parcel identity, geometry, all dated physical-band/index statistics,
finite-or-null values, reliability bounds, and annual output. Missing observations
remain permissible; the cleaning report records their provenance.

In [ ]:
assert len(result) == len(test_parcels)
assert result[PARCEL_ID_FIELD].is_unique
assert set(result[PARCEL_ID_FIELD]) == set(test_parcels[PARCEL_ID_FIELD].astype(str))
assert result.crs.to_epsg() == WORKING_EPSG
assert result.geometry.notna().all() and not result.geometry.is_empty.any()
variables = [f"B{i}" for i in range(1, 9)] + ["EVI", "NDRE", "NDVI", "NDWI", "SAVI"]
features = [f"{variable}_{stat}__{month.replace('-', '')}" for variable in variables
            for stat in ["mean", "median", "sd", "min", "max", "range"] for month in months]
assert set(features).issubset(result.columns)
assert not np.isinf(result[features].to_numpy(dtype=float)).any()
assert result.data_reliability_score.dropna().between(0, 1).all()
annual = gpd.read_parquet(extractor.output_dir / extractor.REDUCED_PARCEL_OUTPUT_FILE_NAME)
assert len(annual) == len(result)
summary = {
    "status": "passed", "test_scope": "all_covered" if TEST_ALL_COVERED else "representative_sample", "tile_months": len(manifest), "quads": len(QUAD_IDS), "months": len(months),
    "input_parcels": len(parcels), "covered_parcels": len(eligible), "outside_coverage": int((~covered).sum()),
    "tested_parcels": len(result), "seam_groups": len(seam_groups), "output_columns": len(result.columns),
    "checked_dated_features": len(features), "elapsed_seconds": elapsed_seconds,
    "ndvi_reference_mean_absolute_error": ndvi_error,
    "reliability_min": float(result.data_reliability_score.min()),
    "reliability_max": float(result.data_reliability_score.max()),
    "dated_nonnull_fraction": float(result[features].notna().to_numpy().mean()),
    "result_path": str(extractor.output_dir / extractor.PARCEL_OUTPUT_FILE_NAME),
}
(OUTPUT_DIR / "validation_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))